# 27a — Transformers and Attention Mechanism

Corresponds to: Note 27 — Transformers and Attention Mechanism

**Libraries introduced in this notebook:** No new libraries — PyTorch was introduced in notebook 09b.

**What this notebook demonstrates:**
1. Manual scaled dot-product attention — reproducing the worked example from Note 27
2. PyTorch's `scaled_dot_product_attention` — verifying it matches the manual result
3. Sinusoidal positional encoding — implementing the formula and inspecting the pattern
4. Transformer encoder layer in PyTorch — `nn.MultiheadAttention` + FFN + residual + LayerNorm

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

torch.manual_seed(0)
print(f'PyTorch: {torch.__version__}')

PyTorch: 2.14.1+cpu


## Part 1: Manual Scaled Dot-Product Attention

Reproducing the Note 27 worked example exactly.
Three key-value positions, one query, $d_k = d_v = 2$.

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

In [2]:
K = np.array([[1.0, 0.0],
              [0.5, 0.5],
              [0.0, 1.0]])   # (3, 2)

V = K.copy()                 # K = V in this example

q = np.array([[0.9, 0.1]])   # (1, 2)

d_k = K.shape[1]

# Step 1: raw scores
scores = q @ K.T             # (1, 3)
print(f'Raw scores  qKᵀ       = {scores.ravel()}')

# Step 2: scale
scores_scaled = scores / np.sqrt(d_k)
print(f'Scaled      / √{d_k}      = {scores_scaled.ravel().round(3)}')

# Step 3: softmax
e = np.exp(scores_scaled - scores_scaled.max())   # numerically stable
alpha = e / e.sum()
print(f'Weights     softmax   = {alpha.ravel().round(3)}')

# Step 4: weighted sum of values
out = alpha @ V              # (1, 2)
print(f'Output      αV        = {out.ravel().round(3)}')
print()
print('Matches Note 27: α = [0.430, 0.325, 0.245]  out = [0.593, 0.408]')

Raw scores  qKᵀ       = [0.9 0.5 0.1]
Scaled      / √2      = [0.636 0.354 0.071]
Weights     softmax   = [0.431 0.325 0.245]
Output      αV        = [0.593 0.407]

Matches Note 27: α = [0.430, 0.325, 0.245]  out = [0.593, 0.408]


## Part 2: PyTorch `scaled_dot_product_attention`

PyTorch provides `F.scaled_dot_product_attention` which implements the same formula. Verifying it matches the manual result.

In [3]:
# PyTorch expects (batch, heads, seq, dim) or (batch, seq, dim)
# Using (1, 1, seq, dim) — batch=1, heads=1
Q_t = torch.tensor(q,   dtype=torch.float32).unsqueeze(0).unsqueeze(0)  # (1,1,1,2)
K_t = torch.tensor(K,   dtype=torch.float32).unsqueeze(0).unsqueeze(0)  # (1,1,3,2)
V_t = torch.tensor(V,   dtype=torch.float32).unsqueeze(0).unsqueeze(0)  # (1,1,3,2)

out_pt = F.scaled_dot_product_attention(Q_t, K_t, V_t)

print(f'PyTorch output: {out_pt.squeeze().tolist()}')
print(f'Manual output:  {out.ravel().round(4).tolist()}')
print(f'Max difference: {abs(out_pt.squeeze().numpy() - out.ravel()).max():.2e}')

PyTorch output: [0.593045175075531, 0.4069547653198242]
Manual output:  [0.593, 0.407]
Max difference: 5.41e-08


In [4]:
# Show how attention weights change with a different query
queries = [
    ([0.9, 0.1], 'similar to key 1 [1.0, 0.0]'),
    ([0.1, 0.9], 'similar to key 3 [0.0, 1.0]'),
    ([0.5, 0.5], 'equidistant from all keys'),
]

print(f'{"Query":<20} {"α₁":>8} {"α₂":>8} {"α₃":>8}  Description')
print('-' * 70)
for qv, desc in queries:
    q_test = np.array([qv])
    s = q_test @ K.T / np.sqrt(d_k)
    e = np.exp(s - s.max())
    a = (e / e.sum()).ravel()
    print(f'{str(qv):<20} {a[0]:>8.3f} {a[1]:>8.3f} {a[2]:>8.3f}  {desc}')

Query                      α₁       α₂       α₃  Description
----------------------------------------------------------------------
[0.9, 0.1]              0.431    0.325    0.245  similar to key 1 [1.0, 0.0]
[0.1, 0.9]              0.245    0.325    0.431  similar to key 3 [0.0, 1.0]
[0.5, 0.5]              0.333    0.333    0.333  equidistant from all keys


## Part 3: Sinusoidal Positional Encoding

Implementing the formula from Note 27:

$$\text{PE}(t, 2i) = \sin\!\left(\frac{t}{10000^{2i/d_{\text{model}}}}\right) \qquad \text{PE}(t, 2i+1) = \cos\!\left(\frac{t}{10000^{2i/d_{\text{model}}}}\right)$$

In [5]:
def sinusoidal_pe(max_len, d_model):
    pe = np.zeros((max_len, d_model))
    positions = np.arange(max_len)[:, None]          # (max_len, 1)
    dims = np.arange(0, d_model, 2)                  # even indices
    div = np.power(10000.0, dims / d_model)           # (d_model/2,)
    pe[:, 0::2] = np.sin(positions / div)             # even dims
    pe[:, 1::2] = np.cos(positions / div)             # odd dims
    return pe

T, d_model = 8, 16
PE = sinusoidal_pe(T, d_model)

print(f'Positional encoding shape: {PE.shape}  (T={T}, d_model={d_model})')
print()
print('PE values for first 4 positions, first 6 dimensions:')
print(f'{"pos":<5}', end='')
for d in range(6):
    print(f'  dim{d}  ', end='')
print()
for t in range(4):
    print(f'{t:<5}', end='')
    for d in range(6):
        print(f'  {PE[t, d]:+.4f}', end='')
    print()
print()
print('Each position has a unique encoding.')
print('Adjacent positions have similar encodings (smooth variation).')

Positional encoding shape: (8, 16)  (T=8, d_model=16)

PE values for first 4 positions, first 6 dimensions:
pos    dim0    dim1    dim2    dim3    dim4    dim5  
0      +0.0000  +1.0000  +0.0000  +1.0000  +0.0000  +1.0000
1      +0.8415  +0.5403  +0.3110  +0.9504  +0.0998  +0.9950
2      +0.9093  -0.4161  +0.5911  +0.8066  +0.1987  +0.9801
3      +0.1411  -0.9900  +0.8126  +0.5828  +0.2955  +0.9553

Each position has a unique encoding.
Adjacent positions have similar encodings (smooth variation).


In [6]:
# Verify: each position is unique
diffs = []
for i in range(T - 1):
    d = np.linalg.norm(PE[i] - PE[i+1])
    diffs.append(d)

print('L2 distance between adjacent position encodings:')
for i, d in enumerate(diffs):
    print(f'  ||PE({i}) - PE({i+1})|| = {d:.4f}')
print()
print('All distances are non-zero — every position has a distinct encoding.')

L2 distance between adjacent position encodings:
  ||PE(0) - PE(1)|| = 1.0147
  ||PE(1) - PE(2)|| = 1.0147
  ||PE(2) - PE(3)|| = 1.0147
  ||PE(3) - PE(4)|| = 1.0147
  ||PE(4) - PE(5)|| = 1.0147
  ||PE(5) - PE(6)|| = 1.0147
  ||PE(6) - PE(7)|| = 1.0147

All distances are non-zero — every position has a distinct encoding.


## Part 4: Transformer Encoder Layer in PyTorch

Building one Transformer encoder layer:
- Multi-head self-attention (`nn.MultiheadAttention`)
- Residual connection + layer normalisation
- Feed-forward network (two-layer MLP)
- Residual connection + layer normalisation

In [7]:
class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        # Sub-layer 1: self-attention + residual + norm
        attn_out, attn_weights = self.self_attn(x, x, x)
        x = self.norm1(x + attn_out)
        # Sub-layer 2: FFN + residual + norm
        x = self.norm2(x + self.ffn(x))
        return x, attn_weights

d_model, n_heads, d_ff = 32, 4, 64
layer = TransformerEncoderLayer(d_model, n_heads, d_ff)

total = sum(p.numel() for p in layer.parameters())
print(f'Encoder layer: d_model={d_model}, n_heads={n_heads}, d_ff={d_ff}')
print(f'Total parameters: {total}')
print(f'  d_k per head = d_model / n_heads = {d_model} / {n_heads} = {d_model // n_heads}')

Encoder layer: d_model=32, n_heads=4, d_ff=64
Total parameters: 8544
  d_k per head = d_model / n_heads = 32 / 4 = 8


In [8]:
# Forward pass: batch=2, seq_len=6, d_model=32
batch, seq_len = 2, 6
x = torch.randn(batch, seq_len, d_model)

out, weights = layer(x)

print(f'Input  shape: {x.shape}   (batch, seq_len, d_model)')
print(f'Output shape: {out.shape}  (batch, seq_len, d_model) — same shape as input')
print(f'Attn weights: {weights.shape}  (batch, seq_len, seq_len) — each position attends to all')
print()
print('Attention weight matrix (batch 0) — rows are queries, columns are keys:')
print(weights[0].detach().numpy().round(3))
print()
print('Each row sums to 1.0 (softmax over keys):')
print(weights[0].sum(dim=-1).detach().numpy().round(4))

Input  shape: torch.Size([2, 6, 32])   (batch, seq_len, d_model)
Output shape: torch.Size([2, 6, 32])  (batch, seq_len, d_model) — same shape as input
Attn weights: torch.Size([2, 6, 6])  (batch, seq_len, seq_len) — each position attends to all

Attention weight matrix (batch 0) — rows are queries, columns are keys:
[[0.201 0.245 0.167 0.126 0.134 0.128]
 [0.109 0.244 0.181 0.207 0.118 0.141]
 [0.16  0.173 0.158 0.204 0.161 0.144]
 [0.168 0.143 0.189 0.128 0.184 0.188]
 [0.167 0.177 0.199 0.1   0.153 0.204]
 [0.19  0.142 0.124 0.198 0.164 0.183]]

Each row sums to 1.0 (softmax over keys):
[1. 1. 1. 1. 1. 1.]


In [9]:
# Compare: PyTorch's built-in TransformerEncoderLayer
builtin = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads,
                                      dim_feedforward=d_ff, batch_first=True)
builtin_params = sum(p.numel() for p in builtin.parameters())
print(f'Our implementation:      {total} parameters')
print(f'nn.TransformerEncoderLayer: {builtin_params} parameters')
print('(Difference: nn.TransformerEncoderLayer includes dropout parameters)')

Our implementation:      8544 parameters
nn.TransformerEncoderLayer: 8544 parameters
(Difference: nn.TransformerEncoderLayer includes dropout parameters)


## Summary

What this notebook demonstrated:

1. **Manual attention** — reproduced the Note 27 worked example. Query $q=[0.9, 0.1]$ against three keys produces weights $[0.430, 0.325, 0.245]$ and output $[0.593, 0.408]$. A query similar to key 1 concentrates weight on position 1; a query similar to key 3 concentrates weight on position 3 — attention is query-dependent.
2. **`F.scaled_dot_product_attention`** — PyTorch's implementation matches the manual result to floating-point precision. The function handles the scaling, softmax, and weighted sum in one call.
3. **Sinusoidal positional encoding** — each position receives a unique encoding from the sinusoidal formula. Adjacent positions have similar encodings (smooth variation), and all pairwise distances are non-zero.
4. **Transformer encoder layer** — built from `nn.MultiheadAttention` + FFN + two residual+norm sub-layers. Output shape equals input shape `(batch, seq_len, d_model)`. The attention weight matrix has shape `(batch, seq_len, seq_len)` — every position attends to every other position, and each row sums to 1.